# Count-based reco binnings

In [ ]:
from datetime import date
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'xml' / 'binning_study').is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/xml/binning_study')

INPUT_FILE = '/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root'
MC_POT, DATA_POT = 9.3221e19, 1.75e21
POT_SCALE = DATA_POT / MC_POT

## Controls

In [ ]:
Q2_RANGE = (-2.0, 0.2)          # log10(Q^2 / GeV^2), the production acceptance
PN_RANGE = (0.0, 1.0)           # GeV
ROUND_Q2, ROUND_PN = 0.01, 0.025
LADDER = (100, 50, 25, 10, 5)   # minimum raw MC events per bin, one variant per rung
RESOLUTION_FACTOR = 1
PROTECTED_Q2, PROTECTED_PN = (), ()    # edges always present, e.g. PROTECTED_PN = (0.3,)
VARIANT_PREFIX = 'count'
OUTPUT_JSON = REPO / 'xml' / 'binning_study' / 'count_variants.json'

# Reference grids drawn for comparison.
NOMINAL_Q2 = [-2.00, -1.50, -1.20, -1.00, -0.85, -0.70, -0.55, -0.40, -0.20, 0.20]
NOMINAL_PN = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.00]

In [ ]:
tree = uproot.open(INPUT_FILE)['tree']
arrays = tree.arrays(
    ['isdata', 'isext', 'isdirt', 'isnuwro', 'afro_1mu1p_sel', 'afro_1mu1p_true',
     'afro_1mu1p_Q2', 'afro_1mu1p_Pn', 'afro_1mu1p_true_Pn', 'GTruth_gQ2', 'net_weight'],
    library='np',
)
overlay = ((arrays['isdata'] == 0) & (arrays['isext'] == 0) & (arrays['isdirt'] == 0)
           & (arrays['isnuwro'] == 0) & (arrays['afro_1mu1p_sel'] == 1))
signal = overlay & (arrays['afro_1mu1p_true'] == 1)
nuwro = (arrays['isnuwro'] == 1) & (arrays['afro_1mu1p_sel'] == 1)
with np.errstate(divide='ignore', invalid='ignore'):
    LOGQ2 = np.log10(arrays['afro_1mu1p_Q2'])
    LOGQ2_TRUE = np.log10(arrays['GTruth_gQ2'])
PN = arrays['afro_1mu1p_Pn']
PN_TRUE = arrays['afro_1mu1p_true_Pn']   # -9999 unless afro_1mu1p_true == 1
EXPECTED_W = arrays['net_weight'] * POT_SCALE

X, Y = LOGQ2[overlay], PN[overlay]            # the sample every count refers to
in_range = (X >= Q2_RANGE[0]) & (X < Q2_RANGE[1]) & (Y >= PN_RANGE[0]) & (Y < PN_RANGE[1])
print(f'selected overlay events: {overlay.sum()}, inside the acceptance: {in_range.sum()}')

## Section 1: reco resolution on both axes

Signal events only, since the true quantities are only meaningful for them. In slices of the
reco variable the resolution is half the 16-84% spread of true minus reco, for log10(Q^2)
against GENIE's `GTruth_gQ2` and for p_n against `afro_1mu1p_true_Pn` (the STV built from the
true muon and proton, so it measures the reconstruction alone, not the FSI smearing that
separates p_n from the initial nucleon momentum). Bins narrower than `RESOLUTION_FACTOR` times
this are not allowed, because neighbouring bins that migrate into each other carry little
independent information.


In [ ]:
def slice_resolution(reco, true, edges, min_events=50):
    '''Per-slice (centre, half 16-84% spread, median bias) of true - reco, in slices of reco.'''
    residual = true - reco
    centers, sigmas, biases = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        part = residual[(reco >= lo) & (reco < hi)]
        if len(part) < min_events:
            continue
        q16, q50, q84 = np.percentile(part, [16, 50, 84])
        centers.append(0.5 * (lo + hi)); sigmas.append(0.5 * (q84 - q16)); biases.append(q50)
    return tuple(map(np.asarray, (centers, sigmas, biases)))


Q2_RES = slice_resolution(LOGQ2[signal], LOGQ2_TRUE[signal], np.arange(Q2_RANGE[0], Q2_RANGE[1] + 1e-9, 0.05))
PN_RES = slice_resolution(PN[signal], PN_TRUE[signal], np.arange(PN_RANGE[0], PN_RANGE[1] + 1e-9, 0.05))


def resolution_at_q2(x):
    '''Reco log10(Q^2) resolution, interpolated between slice centres and held constant outside.'''
    return np.interp(x, Q2_RES[0], Q2_RES[1])


def resolution_at_pn(y):
    '''Reco p_n resolution (GeV), interpolated between slice centres and held constant outside.'''
    return np.interp(y, PN_RES[0], PN_RES[1])


fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist2d(LOGQ2[signal], LOGQ2_TRUE[signal], bins=[np.linspace(*Q2_RANGE, 60), np.linspace(*Q2_RANGE, 60)], cmap='Blues', norm=mpl.colors.LogNorm())
axes[0, 0].plot(Q2_RANGE, Q2_RANGE, 'k--', lw=0.8)
axes[0, 0].set_xlabel('reco $\\log_{10}Q^2$'); axes[0, 0].set_ylabel('true $\\log_{10}Q^2$ (GENIE)')
axes[0, 1].plot(Q2_RES[0], Q2_RES[1], 'o-', label='resolution (half 16-84% spread)')
axes[0, 1].plot(Q2_RES[0], Q2_RES[2], 's--', label='median bias (true - reco)')
axes[0, 1].axhline(0, color='grey', lw=0.8)
axes[0, 1].set_xlabel('reco $\\log_{10}Q^2$'); axes[0, 1].set_ylabel('$\\log_{10}Q^2$'); axes[0, 1].legend()
axes[1, 0].hist2d(PN[signal], PN_TRUE[signal], bins=[np.linspace(*PN_RANGE, 60), np.linspace(*PN_RANGE, 60)], cmap='Blues', norm=mpl.colors.LogNorm())
axes[1, 0].plot(PN_RANGE, PN_RANGE, 'k--', lw=0.8)
axes[1, 0].set_xlabel('reco $p_n$ [GeV]'); axes[1, 0].set_ylabel('true $p_n$ [GeV]')
axes[1, 1].plot(PN_RES[0], PN_RES[1], 'o-', label='resolution (half 16-84% spread)')
axes[1, 1].plot(PN_RES[0], PN_RES[2], 's--', label='median bias (true - reco)')
axes[1, 1].axhline(0, color='grey', lw=0.8)
axes[1, 1].set_xlabel('reco $p_n$ [GeV]'); axes[1, 1].set_ylabel('$p_n$ [GeV]'); axes[1, 1].legend()
fig.tight_layout(); plt.show()
display(pd.DataFrame({'reco log10Q2 centre': Q2_RES[0], 'resolution': Q2_RES[1], 'median bias': Q2_RES[2]}).round(3).T)
display(pd.DataFrame({'reco p_n centre [GeV]': PN_RES[0], 'resolution [GeV]': PN_RES[1], 'median bias [GeV]': PN_RES[2]}).round(3).T)
fig.savefig('/nevis/riverside/share/epelaez/axial_mass/ma_zexp/figs/resolution.pdf', dpi=300, bbox_inches='tight')

## Section 2: grow the ladder

In [ ]:
def counts(q2_edges, pn_edges, mask=None, weights=None):
    x, y = (X, Y) if mask is None else (LOGQ2[mask], PN[mask])
    return np.histogram2d(x, y, bins=[q2_edges, pn_edges], weights=weights)[0]


def round_to(value, step):
    return float(np.round(np.round(value / step) * step, 6))


def width_ok(lo, edge, hi, factor, resolution_at):
    '''Both halves of a split must be at least `factor` times the local reco resolution wide.'''
    for a, b in ((lo, edge), (edge, hi)):
        if (b - a) < factor * resolution_at(0.5 * (a + b)):
            return False
    return True


RESOLUTION_AT = {'q2': resolution_at_q2, 'pn': resolution_at_pn}


def grow(n_min, factor=RESOLUTION_FACTOR, protected_q2=PROTECTED_Q2, protected_pn=PROTECTED_PN, verbose=False):
    '''Greedy equal-population splitting: split the most populated admissible interval until none is left.'''
    q2 = sorted({*Q2_RANGE, *protected_q2})
    pn = sorted({*PN_RANGE, *protected_pn})
    history = []
    while True:
        candidates = []
        for axis, edges, values, step in (('q2', q2, X, ROUND_Q2), ('pn', pn, Y, ROUND_PN)):
            for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
                inside = values[(values >= lo) & (values < hi)]
                if len(inside) == 0:
                    continue
                edge = round_to(np.median(inside), step)
                if not lo < edge < hi:
                    continue
                if not width_ok(lo, edge, hi, factor, RESOLUTION_AT[axis]):
                    continue
                # most populated interval first; ties go to the axis with fewer bins
                candidates.append((len(inside), -(len(edges) - 1), axis, edge))
        candidates.sort(reverse=True)
        accepted = None
        for population, _, axis, edge in candidates:
            trial = sorted((q2 if axis == 'q2' else pn) + [edge])
            h = counts(trial, pn) if axis == 'q2' else counts(q2, trial)
            if h.min() >= n_min:
                accepted = (axis, edge, h.min(), population)
                break
        if accepted is None:
            break
        axis, edge, hmin, population = accepted
        if axis == 'q2':
            q2 = sorted(q2 + [edge])
        else:
            pn = sorted(pn + [edge])
        history.append(dict(step=len(history) + 1, axis=axis, edge=edge, interval_events=int(population),
                            nQ2=len(q2) - 1, npn=len(pn) - 1, min_cell=int(hmin)))
        if verbose:
            print(f'  step {len(history):2d}: split {axis} at {edge:g} ({population} events in the interval)'
                  f' -> {len(q2) - 1} x {len(pn) - 1}, min cell {hmin:.0f}')
    return q2, pn, pd.DataFrame(history)


def fmt_edges(edges):
    def fmt(v):
        text = f'{v:.3f}'
        return text[:-1] if text.endswith('0') else text
    return ' '.join(fmt(e) for e in edges)


def migration_frac(reco, true, edges):
    '''Fraction of signal events whose true value lands in a different bin than reco, for one
    axis's edges.  This is what the resolution guard is trying to control; unlike MC min it
    responds to bin PLACEMENT, not just bin count, so it catches guard floors that are too
    lax (bins barely wider than 1 sigma still migrate heavily) independent of statistics.'''
    edges = np.asarray(edges)
    reco_bin = np.digitize(reco, edges) - 1
    true_bin = np.digitize(true, edges) - 1
    valid = (reco_bin >= 0) & (reco_bin < len(edges) - 1) & (true_bin >= 0) & (true_bin < len(edges) - 1)
    return float(np.mean(reco_bin[valid] != true_bin[valid])) if valid.any() else float('nan')


def min_width_over_resolution(edges, resolution_at):
    '''Smallest bin-width / local-resolution ratio over a 1D grid, i.e. how close the tightest
    bin sits to the guard floor.  A value near RESOLUTION_FACTOR means that bin is barely
    admissible and should be read together with its migration fraction, not on its own.'''
    edges = np.asarray(edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)
    return float(np.min(widths / resolution_at(centers)))


def summarize(name, q2, pn, n_min=None):
    h = counts(q2, pn)
    expected = counts(q2, pn, overlay, EXPECTED_W[overlay])
    nw = counts(q2, pn, nuwro)
    return {'variant': name, 'N_min': n_min, 'nQ2': len(q2) - 1, 'npn': len(pn) - 1, 'nbins': h.size,
            'MC min': int(h.min()), 'MC median': float(np.median(h)), 'MC max': int(h.max()),
            'max/min': float(h.max() / max(h.min(), 1)),
            'expected min': float(expected.min()), 'NuWro min': int(nw.min()),
            'p_n migration': migration_frac(PN[signal], PN_TRUE[signal], pn),
            'Q2 migration': migration_frac(LOGQ2[signal], LOGQ2_TRUE[signal], q2),
            'min w/res (q2)': min_width_over_resolution(q2, resolution_at_q2),
            'min w/res (pn)': min_width_over_resolution(pn, resolution_at_pn)}

In [ ]:
LADDER_GRIDS = {}
rows = []
for n_min in LADDER:
    name = f'{VARIANT_PREFIX}{n_min}'
    q2, pn, history = grow(n_min)
    LADDER_GRIDS[name] = dict(q2=q2, pn=pn, n_min=n_min, history=history)
    rows.append(summarize(name, q2, pn, n_min))
    print(f'{name:10s} {len(q2) - 1:2d} x {len(pn) - 1:2d} = {(len(q2) - 1) * (len(pn) - 1):3d} bins')
    print(f'{"":10s} log10Q2: {fmt_edges(q2)}')
    print(f'{"":10s} p_n:     {fmt_edges(pn)}')
rows.append(summarize('nominal (reference)', NOMINAL_Q2, NOMINAL_PN))
SUMMARY = pd.DataFrame(rows).set_index('variant')
display(SUMMARY.round(1))

The order in which edges were accepted for one rung, with the population of the interval that
was split. Early splits halve the fattest intervals; the last ones are where the occupancy
constraint bites, and the axis that stops first tells you which direction the sparse corner
limits.

In [ ]:
SHOW_HISTORY = f'{VARIANT_PREFIX}100'
display(LADDER_GRIDS[SHOW_HISTORY]['history'])

## Section 3: the grids

Each panel colours the cells by their raw MC count and annotates them when they are wide enough
to read. The nominal grid is shown for comparison.

In [ ]:
def draw_grid(ax, q2, pn, title, annotate=True, n_min=None):
    '''Cells coloured by raw MC count (log scale) with the grid drawn on top.'''
    h = counts(q2, pn)
    mesh = ax.pcolormesh(pn, q2, h, cmap='viridis', norm=mpl.colors.LogNorm(vmin=max(h.min(), 1), vmax=h.max()),
                         edgecolors='white', linewidth=0.4)
    plt.colorbar(mesh, ax=ax, label='raw MC events')
    if annotate and h.size <= 300:
        for i in range(len(q2) - 1):
            for j in range(len(pn) - 1):
                if (pn[j + 1] - pn[j]) < 0.03 and h.size > 120:
                    continue   # too narrow to label
                ax.text(0.5 * (pn[j] + pn[j + 1]), 0.5 * (q2[i] + q2[i + 1]), f'{h[i, j]:.0f}',
                        ha='center', va='center', fontsize=5.5, color='white' if h[i, j] < 0.3 * h.max() else 'black')
    ax.set_xlim(*PN_RANGE); ax.set_ylim(*Q2_RANGE)
    ax.set_xlabel('$p_n$ [GeV]'); ax.set_ylabel('$\\log_{10}(Q^2/\\mathrm{GeV}^2)$')
    ax.set_title(f'{title}: {len(q2) - 1} x {len(pn) - 1} = {h.size} bins, min {h.min():.0f}' + (f' (N_min {n_min})' if n_min else ''), fontsize=9)


panels = [('nominal', NOMINAL_Q2, NOMINAL_PN, None)] + [(name, g['q2'], g['pn'], g['n_min']) for name, g in LADDER_GRIDS.items()]
ncol = 3
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 5 * nrow), squeeze=False)
for ax, (name, q2, pn, n_min) in zip(axes.flat, panels):
    draw_grid(ax, q2, pn, name, n_min=n_min)
for ax in axes.flat[len(panels):]:
    ax.axis('off')
fig.tight_layout(); plt.show()
fig.savefig('/nevis/riverside/share/epelaez/axial_mass/ma_zexp/figs/binning_options.pdf', dpi=300, bbox_inches='tight')

Bin widths against the resolution guard on both axes: the bins of each rung compared with the
local resolution. Bins sitting on the guard line are resolution-limited, bins above it are
occupancy-limited.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, axis, rng, nominal, label in ((axes[0], 'q2', Q2_RANGE, NOMINAL_Q2, 'reco $\\log_{10}Q^2$'),
                                      (axes[1], 'pn', PN_RANGE, NOMINAL_PN, 'reco $p_n$ [GeV]')):
    xx = np.linspace(*rng, 200)
    ax.plot(xx, RESOLUTION_FACTOR * RESOLUTION_AT[axis](xx), 'k--', label=f'{RESOLUTION_FACTOR:g} x resolution')
    for name, g in LADDER_GRIDS.items():
        e = np.asarray(g[axis])
        ax.step(e[:-1], np.diff(e), where='post', label=name, alpha=0.8)
    ne = np.asarray(nominal)
    ax.step(ne[:-1], np.diff(ne), where='post', color='grey', lw=2, label='nominal')
    ax.set_xlabel(f'{label} (bin low edge)'); ax.set_ylabel('bin width'); ax.set_yscale('log'); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


## Section 4: hand the ladder to the generator

`count_variants.json` is read by `make_binning_study_xmls.py`; rerun it (with `--check`) to
write the XMLs, then `xml/binning_study/run_all.sh --variant count100 ...` runs them and
notebook 16 picks them up like any other variant.

In [ ]:
payload = {
    '_meta': dict(
        generated=str(date.today()), notebook='python/notebooks/17_count_based_binning.ipynb',
        rule='greedy equal-population splitting, min raw MC events per bin = N_min',
        q2_range=list(Q2_RANGE), pn_range=list(PN_RANGE), round_q2=ROUND_Q2, round_pn=ROUND_PN,
        resolution_factor=RESOLUTION_FACTOR, protected_q2=list(PROTECTED_Q2), protected_pn=list(PROTECTED_PN),
    ),
}
for name, g in LADDER_GRIDS.items():
    nq, npn = len(g['q2']) - 1, len(g['pn']) - 1
    payload[name] = dict(
        q2=[float(e) for e in g['q2']], pn=[float(e) for e in g['pn']],
        description=f'count-based grid, >= {g["n_min"]} raw MC events per bin ({nq} x {npn} = {nq * npn} bins)',
    )
OUTPUT_JSON.write_text(json.dumps(payload, indent=1) + '\n')
print(f'wrote {len(LADDER_GRIDS)} variants to {OUTPUT_JSON}')
print('next: python/scripts/make_binning_study_xmls.py --check --variant ' + ' --variant '.join(LADDER_GRIDS))